# SPX evidence, Heston calibration, and held-out validation

This study connects three already-earned layers without moving quantitative logic into the notebook:

1. M4 observation/normalization and implied-volatility evidence,
2. M6 calibration and an identifiability counterexample,
3. M7 Black–Scholes versus Heston same-date cross-sectional validation.

Raw vendor/source rows are intentionally not redistributed; the default path uses package-safe derived evidence. See [`m4_market_evidence_and_implied_volatility.md`](../docs/models/m4_market_evidence_and_implied_volatility.md), [`m6_heston_calibration.md`](../docs/models/m6_heston_calibration.md), and [`m7_empirical_validation_and_model_risk.md`](../docs/models/m7_empirical_validation_and_model_risk.md).


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from qf_platform.application import (
    HestonCalibrationWorkbenchRequest,
    canonical_m4_market_workbench,
    make_ui5_reference_validation_request,
    run_heston_calibration,
    run_ui5_validation,
)
from qf_platform.inference import HestonCalibrationCoordinates
from qf_platform.pricing import HestonParameters
from qf_platform.validation import ValidationModel, ValidationPartition

market = canonical_m4_market_workbench()

In [ ]:
def show_table(headers, rows):
    text = "| " + " | ".join(headers) + " |\n"
    text += "| " + " | ".join("---" for _ in headers) + " |\n"
    for row in rows:
        text += "| " + " | ".join(str(value) for value in row) + " |\n"
    display(Markdown(text))


outcome = market.outcomes[0]
assert outcome.normalized is not None
assert outcome.result is not None
show_table(
    ("stage", "value", "meaning"),
    (
        (
            "raw bid / ask",
            f"{outcome.raw_quote.bid} / {outcome.raw_quote.ask}",
            "observed synthetic quote",
        ),
        (
            "normalized target",
            f"{outcome.normalized.target_price:.6f}",
            "explicit midpoint construction",
        ),
        (
            "implied volatility",
            f"{outcome.result.annualized_volatility:.6f}",
            "model-dependent inverse result",
        ),
    ),
)

In [ ]:
expiries = sorted({point.expiry for point in market.empirical_evidence.points})
plt.figure(figsize=(7, 4))
for expiry in expiries:
    points = [
        point for point in market.empirical_evidence.points if point.expiry == expiry
    ]
    points = sorted(points, key=lambda point: point.strike)
    plt.plot(
        [point.strike for point in points],
        [point.annualized_implied_volatility for point in points],
        marker="o",
        label=expiry.isoformat(),
    )
plt.xlabel("strike")
plt.ylabel("annualized implied volatility")
plt.title("Pinned derived SPX smile/skew evidence")
plt.legend()
plt.show()
display(Markdown("**Evidence boundary:** " + market.empirical_evidence.license_note))

In [ ]:
validation = run_ui5_validation(make_ui5_reference_validation_request())
evidence = validation.evidence

bs_train = evidence.black_scholes_training_metrics
heston_train = evidence.heston_training_metrics
bs_eval = evidence.black_scholes_evaluation_metrics
heston_eval = evidence.heston_evaluation_metrics
metric_rows = (
    (
        "training",
        "Black–Scholes",
        f"{bs_train.root_mean_square_error:.3f}",
        f"{100 * bs_train.relative_mean_absolute_error:.2f}%",
        f"{bs_train.standardized_root_mean_square_error:.3f}",
    ),
    (
        "training",
        "Heston",
        f"{heston_train.root_mean_square_error:.3f}",
        f"{100 * heston_train.relative_mean_absolute_error:.2f}%",
        f"{heston_train.standardized_root_mean_square_error:.3f}",
    ),
    (
        "held out",
        "Black–Scholes",
        f"{bs_eval.root_mean_square_error:.3f}",
        f"{100 * bs_eval.relative_mean_absolute_error:.2f}%",
        f"{bs_eval.standardized_root_mean_square_error:.3f}",
    ),
    (
        "held out",
        "Heston",
        f"{heston_eval.root_mean_square_error:.3f}",
        f"{100 * heston_eval.relative_mean_absolute_error:.2f}%",
        f"{heston_eval.standardized_root_mean_square_error:.3f}",
    ),
)
show_table(
    ("partition", "model", "price RMSE", "relative MAE", "standardized RMSE"),
    metric_rows,
)

assert abs(bs_eval.root_mean_square_error - 8.412) < 0.0005
assert abs(heston_eval.root_mean_square_error - 0.671) < 0.0005
assert not evidence.conclusion.temporal_out_of_sample_tested
assert not evidence.conclusion.heston_hedge_comparison_supported

In [ ]:
held_out = [
    item
    for item in evidence.residuals
    if item.partition is ValidationPartition.EVALUATION
]
plt.figure(figsize=(7, 4))
for model in (ValidationModel.BLACK_SCHOLES, ValidationModel.HESTON):
    rows = sorted(
        (item for item in held_out if item.model is model),
        key=lambda item: item.strike,
    )
    plt.scatter(
        [item.strike for item in rows],
        [item.residual for item in rows],
        label=model.value,
    )
plt.axhline(0.0, linewidth=1)
plt.xlabel("strike")
plt.ylabel("model price - observed price")
plt.title("Predeclared held-out contract residuals")
plt.legend()
plt.show()

In [ ]:
stability = evidence.heston_stability
training_condition = (
    f"{stability.training_condition_number:.2f}"
    if stability.training_condition_number is not None
    else "n/a"
)
full_condition = (
    f"{stability.full_sample_condition_number:.2f}"
    if stability.full_sample_condition_number is not None
    else "n/a"
)
show_table(
    ("evidence", "training", "full sample"),
    (
        (
            "Jacobian rank",
            stability.training_jacobian_rank,
            stability.full_sample_jacobian_rank,
        ),
        ("condition number", training_condition, full_condition),
    ),
)
print(
    "Maximum train-to-full domain-scaled parameter shift:",
    f"{stability.maximum_train_to_full_domain_scaled_shift:.4f}",
)

In [ ]:
initial_guess = HestonCalibrationCoordinates(
    initial_variance=0.06,
    parameters=HestonParameters(
        mean_reversion_speed=1.2,
        long_run_variance=0.06,
        volatility_of_variance=0.8,
        correlation=-0.4,
        continuous_dividend_yield=0.01,
    ),
)
thin = run_heston_calibration(
    HestonCalibrationWorkbenchRequest("thin", initial_guess, 250, 128)
)

thin_rows = []
for run in thin.runs:
    result = run.result
    thin_rows.append(
        (
            run.label,
            f"{result.objective_value:.3e}",
            result.conditioning.jacobian_rank,
            result.conditioning.rank_deficient,
            tuple(round(value, 5) for value in result.estimate.as_vector()),
        )
    )
show_table(
    ("start", "objective", "rank", "rank deficient", "estimated coordinates"),
    thin_rows,
)

assert all(run.result.conditioning.rank_deficient for run in thin.runs)
assert thin.runs[0].result.estimate != thin.runs[1].result.estimate

## Bounded interpretation

The M7 result is a **predeclared same-date cross-sectional** 10-training / 4-held-out comparison on selected January 4, 2023 SPX/SPXW observations. It is not temporal out-of-sample forecasting and is not a historical trading-profit result. It is also not physical-measure prediction, a global-identification proof, or evidence of Heston hedge superiority.

The deliberately thin M6 calibration above also shows why a very small objective and optimizer convergence do not imply identified parameters: materially different estimates can fit an underdetermined sample while the local Jacobian remains rank deficient.


In [ ]:
display(Markdown("**Authoritative M7 conclusion:** " + evidence.conclusion.statement))